In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:

train_df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test_df = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
submission_df = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')


In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
train_df.isna().sum()

In [ ]:
train_df.duplicated().sum()

In [ ]:
train_df.shape

In [ ]:
train_df.drop_duplicates(inplace=True)

In [ ]:
train_df.shape

In [ ]:
texts = train_df['text'].tolist()
labels = train_df['target'].tolist()

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, stratify=labels, random_state=42)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')


In [ ]:
class DisasterDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = DisasterDataset(train_texts, train_labels, tokenizer)
val_dataset = DisasterDataset(val_texts, val_labels, tokenizer)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
class FrozenBERTClassifier(nn.Module):
    def __init__(self):
        super(FrozenBERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        for param in self.bert.parameters():
            param.requires_grad = False  # تجميد BERT
        self.classifier = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FrozenBERTClassifier().to(device)
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
from tqdm import tqdm

for epoch in range(5):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} - Training loss: {total_loss / len(train_loader)}")

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds))

2- BERT as Embedder + External Classifier


In [ ]:
bert = BertModel.from_pretrained('bert-base-uncased')
bert = bert.to(device)
bert.eval()

In [ ]:
def get_cls_embeddings(texts):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), 16)):
        batch = texts[i:i+16]
        encodings = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=128)
        input_ids = encodings['input_ids'].to(device)
        attention_mask = encodings['attention_mask'].to(device)

        with torch.no_grad():
            outputs = bert(input_ids=input_ids, attention_mask=attention_mask)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # نأخذ فقط [CLS]
            all_embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

In [ ]:
X_train = get_cls_embeddings(train_texts)
X_val = get_cls_embeddings(val_texts)
y_train = np.array(train_labels)
y_val = np.array(val_labels)

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_val)


In [ ]:
print(classification_report(y_val, y_pred))

3-  BERT + Cosine Similarity Classification

In [ ]:
def get_cls_embedding(text):
    tokens = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    input_ids = tokens['input_ids'].to(device)
    attention_mask = tokens['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS]
        return cls_embedding.cpu().numpy()

In [ ]:
bert = BertModel.from_pretrained('bert-base-uncased')
bert = bert.to(device)
bert.eval()

In [ ]:
label_descriptions = {
    0: "This tweet is not about a real disaster.",
    1: "This tweet is about a real disaster."
}

In [ ]:
label_embeddings = {
    label: get_cls_embedding(text)[0]
    for label, text in label_descriptions.items()
}

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

preds = []

for text in tqdm(val_texts):
    text_emb = get_cls_embedding(text)[0].reshape(1, -1)

    sims = {
        label: cosine_similarity(text_emb, emb.reshape(1, -1))[0][0]
        for label, emb in label_embeddings.items()
    }

    pred_label = max(sims, key=sims.get)
    preds.append(pred_label)

In [ ]:
print(classification_report(val_labels, preds))